We will analyze the Porto Seguro's Safe Driver dataset. The goal is to predict whether a driver will make an insurance claim next year.

The evaulation metric: Normalized Gini Coefficient. The code for comupting the metric was taken from the Kaggle competition site.

The dataset contains 595,212 samples over 57 features (all numeric) without missing values. It has a binary traget value of whether an insurance was claimed or not. Since the dataset size is about 34M entries, dimensionally reduction is a crucial step to handle the CPU/Memory load.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import make_scorer
from sklearn.decomposition import PCA
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

In [2]:
def gini(actual, pred, cmpcol = 0, sortcol = 1):
    assert( len(actual) == len(pred) )
    all = np.asarray(np.c_[actual, pred, np.arange(len(actual))], dtype=float)
    all = all[np.lexsort((all[:,2], -1*all[:,1]))]
    totalLosses = all[:,0].sum()
    giniSum = all[:,0].cumsum().sum() / totalLosses

    giniSum -= (len(actual) + 1) / 2.
    return giniSum / len(actual)

def gini_normalized(a, p):
    return gini(a, p) / gini(a, a)

In [3]:
df = pd.read_csv('train.csv')
print("First 5 rows of train:")
print(df.head())

First 5 rows of train:
   id  target  ps_ind_01  ps_ind_02_cat  ps_ind_03  ps_ind_04_cat  \
0   7       0          2              2          5              1   
1   9       0          1              1          7              0   
2  13       0          5              4          9              1   
3  16       0          0              1          2              0   
4  17       0          0              2          0              1   

   ps_ind_05_cat  ps_ind_06_bin  ps_ind_07_bin  ps_ind_08_bin  ...  \
0              0              0              1              0  ...   
1              0              0              0              1  ...   
2              0              0              0              1  ...   
3              0              1              0              0  ...   
4              0              1              0              0  ...   

   ps_calc_11  ps_calc_12  ps_calc_13  ps_calc_14  ps_calc_15_bin  \
0           9           1           5           8               0   
1  

In [4]:
print(df.shape)
print(df.info())
print(df.describe())
print(df['target'].value_counts())


(595212, 59)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 595212 entries, 0 to 595211
Data columns (total 59 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id              595212 non-null  int64  
 1   target          595212 non-null  int64  
 2   ps_ind_01       595212 non-null  int64  
 3   ps_ind_02_cat   595212 non-null  int64  
 4   ps_ind_03       595212 non-null  int64  
 5   ps_ind_04_cat   595212 non-null  int64  
 6   ps_ind_05_cat   595212 non-null  int64  
 7   ps_ind_06_bin   595212 non-null  int64  
 8   ps_ind_07_bin   595212 non-null  int64  
 9   ps_ind_08_bin   595212 non-null  int64  
 10  ps_ind_09_bin   595212 non-null  int64  
 11  ps_ind_10_bin   595212 non-null  int64  
 12  ps_ind_11_bin   595212 non-null  int64  
 13  ps_ind_12_bin   595212 non-null  int64  
 14  ps_ind_13_bin   595212 non-null  int64  
 15  ps_ind_14       595212 non-null  int64  
 16  ps_ind_15       595212 non-null  int64  
 1

21,694/573,518 drivers claimed insurance suggesting imbalanced dataset.

## Data Preprocessing for Modeling

In [20]:
X = df.drop(['id', 'target'], axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Shape of X_train_scaled: {X_train_scaled.shape}")
print(f"Shape of X_test_scaled: {X_test_scaled.shape}")

Shape of X_train_scaled: (476169, 57)
Shape of X_test_scaled: (119043, 57)


## Dimensionality Reduction with PCA and Nyström Kernel PCA

In [21]:
n_components=20

print("\nApplying PCA")
pca = PCA(n_components=n_components, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Shape after PCA: {X_train_pca.shape}")

log_reg_pca = LogisticRegression(random_state=42, solver='liblinear')
log_reg_pca.fit(X_train_pca, y_train)
y_pred_pca = log_reg_pca.predict(X_test_pca)
y_proba_pca = log_reg_pca.predict_proba(X_test_pca)[:, 1]

gini_pca = gini_normalized(y_test, y_proba_pca)

print(f"PCA + Logistic Regression Normalized Gini: {gini_pca:.4f}")


Applying PCA
Shape after PCA: (476169, 20)
PCA + Logistic Regression Normalized Gini: 0.2344


In [22]:
print("\nApplying Nyström Kernel PCA")
nystroem = Nystroem(kernel='rbf', n_components=n_components, random_state=42)
X_train_nystroem = nystroem.fit_transform(X_train_scaled)
X_test_nystroem = nystroem.transform(X_test_scaled)

print(f"Shape after Nyström Kernel PCA: {X_train_nystroem.shape}")

log_reg_nystroem = LogisticRegression(random_state=42, solver='liblinear')
log_reg_nystroem.fit(X_train_nystroem, y_train)
y_pred_nystroem = log_reg_nystroem.predict(X_test_nystroem)
y_proba_nystroem = log_reg_nystroem.predict_proba(X_test_nystroem)[:, 1]

gini_nystroem = gini_normalized(y_test, y_proba_nystroem)

print(f"Nyström Kernel PCA + Logistic Regression Normalized Gini: {gini_nystroem:.4f}")


Applying Nyström Kernel PCA
Shape after Nyström Kernel PCA: (476169, 20)
Nyström Kernel PCA + Logistic Regression Normalized Gini: 0.2221


It looks like that the data is already linear since the kernal PCA performance was a bit worse than the regular PCA. Will continue with the regular PCA.

### 5-Fold Cross-Validation Model Comparison

In [29]:
n_components = 0.8
pca = PCA(n_components=n_components, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
print(f"Shape after PCA: {X_train_pca.shape}")

Shape after PCA: (476169, 36)


In [16]:
def run_cv(model, X, y, model_name, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    gini_scores = []

    print(f"Running 5-Fold Cross-Validation for {model_name}:")
    for fold, (train_index, test_index) in enumerate(skf.split(X, y)):
        X_train_fold, X_test_fold = X[train_index], X[test_index]
        y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]

        model.fit(X_train_fold, y_train_fold)
        y_proba_fold = model.predict_proba(X_test_fold)[:, 1]
        gini = gini_normalized(y_test_fold, y_proba_fold)
        gini_scores.append(gini)
        print(f"Fold {fold+1} Gini: {gini:.4f}")

    avg_gini = np.mean(gini_scores)
    std_gini = np.std(gini_scores)
    print(f"Average Gini for {model_name}: {avg_gini:.4f} (+/- {std_gini:.4f})")
    return avg_gini, std_gini


In [30]:
log_reg_cv = LogisticRegression(random_state=42, solver='liblinear')
avg_gini_log_reg, std_gini_log_reg = run_cv(log_reg_cv, X_train_pca, y_train, 'logistic_reg')


Running 5-Fold Cross-Validation for logistic_reg:
Fold 1 Gini: 0.2336
Fold 2 Gini: 0.2318
Fold 3 Gini: 0.2319
Fold 4 Gini: 0.2233
Fold 5 Gini: 0.2326
Average Gini for logistic_reg: 0.2306 (+/- 0.0037)


In [31]:
xgb_cv = XGBClassifier(objective='binary:logistic', learning_rate=0.01, eval_metric='logloss', random_state=42, n_estimators=100)
avg_gini_xgb, std_gini_xgb = run_cv(xgb_cv, X_train_pca, y_train, 'XGBoost')


Running 5-Fold Cross-Validation for XGBoost:
Fold 1 Gini: 0.2369
Fold 2 Gini: 0.2344
Fold 3 Gini: 0.2378
Fold 4 Gini: 0.2300
Fold 5 Gini: 0.2336
Average Gini for XGBoost: 0.2345 (+/- 0.0028)


In [32]:
lgbm_cv = lgb.LGBMClassifier(objective='binary', learning_rate=0.01, random_state=42, n_estimators=100)
avg_gini_lgbm, std_gini_lgbm = run_cv(lgbm_cv, X_train_pca, y_train, 'LightGBM')


Running 5-Fold Cross-Validation for LightGBM:
[LightGBM] [Info] Number of positive: 13884, number of negative: 367051
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.135806 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9180
[LightGBM] [Info] Number of data points in the train set: 380935, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.036447 -> initscore=-3.274764
[LightGBM] [Info] Start training from score -3.274764


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 1 Gini: 0.2324
[LightGBM] [Info] Number of positive: 13884, number of negative: 367051
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.138094 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9180
[LightGBM] [Info] Number of data points in the train set: 380935, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.036447 -> initscore=-3.274764
[LightGBM] [Info] Start training from score -3.274764


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 2 Gini: 0.2284
[LightGBM] [Info] Number of positive: 13884, number of negative: 367051
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.251810 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9180
[LightGBM] [Info] Number of data points in the train set: 380935, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.036447 -> initscore=-3.274764
[LightGBM] [Info] Start training from score -3.274764


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 3 Gini: 0.2329
[LightGBM] [Info] Number of positive: 13884, number of negative: 367051
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.154044 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9180
[LightGBM] [Info] Number of data points in the train set: 380935, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.036447 -> initscore=-3.274764
[LightGBM] [Info] Start training from score -3.274764


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 4 Gini: 0.2287
[LightGBM] [Info] Number of positive: 13884, number of negative: 367052
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.136908 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9180
[LightGBM] [Info] Number of data points in the train set: 380936, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.036447 -> initscore=-3.274766
[LightGBM] [Info] Start training from score -3.274766


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 5 Gini: 0.2339
Average Gini for LightGBM: 0.2313 (+/- 0.0023)


### Cross-Validation Summary

In [33]:
print("5-Fold Cross-Validation Results")
print(f"Logistic Regression (PCA) Average Gini = {avg_gini_log_reg:.4f} (+/- {std_gini_log_reg:.4f})")
print(f"XGBoost (PCA) Average Gini = {avg_gini_xgb:.4f} (+/- {std_gini_xgb:.4f})")
print(f"LightGBM (PCA) Average Gini = {avg_gini_lgbm:.4f} (+/- {std_gini_lgbm:.4f})")


5-Fold Cross-Validation Results
Logistic Regression (PCA) Average Gini = 0.2306 (+/- 0.0037)
XGBoost (PCA) Average Gini = 0.2345 (+/- 0.0028)
LightGBM (PCA) Average Gini = 0.2313 (+/- 0.0023)


In [36]:
def final_test_report(model, X_train, y_train, X_test, y_test):
  model.fit(X_train, y_train)
  y_proba = model.predict_proba(X_test)[:, 1]
  gini = gini_normalized(y_test, y_proba)
  return gini

In [38]:
gini_log = final_test_report(log_reg_cv, X_train_pca, y_train, X_test_pca, y_test)
gini_xgb = final_test_report(xgb_cv, X_train_pca, y_train, X_test_pca, y_test)
gini_gbm = final_test_report(lgbm_cv, X_train_pca, y_train, X_test_pca, y_test)

[LightGBM] [Info] Number of positive: 17355, number of negative: 458814
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.169299 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9180
[LightGBM] [Info] Number of data points in the train set: 476169, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.036447 -> initscore=-3.274764
[LightGBM] [Info] Start training from score -3.274764


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [39]:
print("Final test performance:")
print(f"Logistic Regression (PCA) Gini: {gini_log:.4f}")
print(f"XGBoost (PCA) Gini: {gini_xgb:.4f}")
print(f"LightGBM (PCA) Gini: {gini_gbm:.4f}")

Final test performance:
Logistic Regression (PCA) Gini: 0.2323
XGBoost (PCA) Gini: 0.2382
LightGBM (PCA) Gini: 0.2367


The best model by 5-fold CV is XGBoost. We will report the results based on this model.

In [43]:
def report_results(model, X_train, y_train, X_test, test_id):
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    res = pd.DataFrame(columns=['id','target'], index=range(len(test_id)))
    res['id'] = test_id
    res['target'] = y_proba
    res = res.sort_values(by='target', ascending=False).reset_index(drop=True)
    res.to_csv('test_results.csv', index=False)
    return res

In [40]:
test = pd.read_csv('test.csv')
test_id = test['id']
test = test.drop('id', axis=1)
test = scaler.transform(test)
test = pca.transform(test)

In [44]:
report_results(xgb_cv, X_train_pca, y_train, test, test_id)

,id,target
0,1112547,0.154303
1,759156,0.142342
2,224989,0.135515
3,1298643,0.133597
4,1173847,0.131799
...,...,...
892811,787366,0.022102
892812,299415,0.022102
892813,504476,0.022076
892814,1041908,0.022047
